# Tutorial 04: Potencial de Coulomb Regularizado (`CoulombPotential`)

En este tutorial simularemos la interacción electrostática de un fermión relativista de Dirac con un centro atractivo de carga nuclear efectiva $Z$ mediante la clase `CoulombPotential(Z, epsilon)`:

$$V(r) = -\frac{Z}{\sqrt{r^2 + \epsilon^2}}$$

donde $\epsilon > 0$ es el parámetro de regularización en el origen ($r = 0$) que evita la divergencia numérica sobre nodos de la malla que coinciden con $r = 0$.

### Objetivos de aprendizaje
1. Analizar el efecto del parámetro de regularización $\epsilon$ y la carga $Z$ en `CoulombPotential(Z, epsilon)`.
2. Simular en **1D** la aceleración y dispersión de un paquete gaussiano al atravesar el centro atractivo de Coulomb.
3. Simular en **2D** la deflexión coulombiana con parámetro de impacto no nulo y visualizar el campo vectorial de corriente de probabilidad $\mathbf{J} = \psi^\dagger \boldsymbol{\alpha} \psi$.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Asegurar que dirac_solver sea localizable desde examples/, examples/es/ o examples/en/
for _rel in ['..', '../..']:
    _abs = os.path.abspath(_rel)
    if os.path.exists(os.path.join(_abs, 'dirac_solver')) and _abs not in sys.path:
        sys.path.insert(0, _abs)

from dirac_solver import (
    Grid,
    ConstantSpinor,
    GaussianPacket,
    PlaneWave,
    DiracProblemBuilder,
    DiracSolver,
    FreeParticle,
    ScalarPotential,
    CoulombPotential,
    YukawaPotential,
    InfiniteWellPotential,
    Observables,
    HDF5Storage,
    _core,
    electron_mass,
    c,
    hbar,
)
from dirac_solver.storage import create_animation, create_animation_1d

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})

---
## 1. Perfil del Potencial de Coulomb Regularizado

Inspeccionemos cómo varía $V(r)$ cerca del origen para distintos valores de $\epsilon$.

In [ ]:
grid_1d = Grid(shape=(320,), spacing=(0.08,))
x = grid_1d.coords[:, 0]

coulomb_eps05 = CoulombPotential(Z=2.0, epsilon=0.5)
coulomb_eps02 = CoulombPotential(Z=2.0, epsilon=0.2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, [coulomb_eps05.evaluate([xi]) for xi in x], 'b-', lw=2, label=r'$Z=2.0, \epsilon=0.5$')
ax.plot(x, [coulomb_eps02.evaluate([xi]) for xi in x], 'r--', lw=2, label=r'$Z=2.0, \epsilon=0.2$')
ax.set_xlabel('Posición x')
ax.set_ylabel('Potencial V(x)')
ax.set_title('Potencial de Coulomb Regularizado en 1D')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Dinámica 1D bajo un Centro Atractivo de Coulomb

Lanzamos un paquete gaussiano desde $x_0 = -4.0$ hacia el centro atractivo en $x = 0$ con momento inicial $p_{0x} = 2.5$ y registramos los valores esperados $\langle x \rangle(t)$, $\langle p_x \rangle(t)$ y $\langle H \rangle(t)$.

In [ ]:
spinor_1d = ConstantSpinor(momentum=[2.5, 0.0, 0.0])
initial_1d = GaussianPacket(spinor_1d, center=[-4.0, 0.0, 0.0], spatial_width=0.9)

problem_1d = (
    DiracProblemBuilder()
    .set_grid(grid_1d)
    .set_initial_state(initial_1d)
    .set_potential(coulomb_eps05)
    .set_boundary_condition(_core.AbsorbingBoundary(strength=0.3, width=25))
    .set_time_parameters(time_step=0.004, total_time=3.6)
    .build()
)

solver_1d = DiracSolver(problem_1d)
obs_1d = Observables(solver_1d)
rho_1d_init = obs_1d.probability_density(solver_1d.get_psi())

solver_1d.run_simulation(observables_handler=obs_1d)
rho_1d_final = obs_1d.probability_density(solver_1d.get_psi())

In [ ]:
t_hist = obs_1d.get_history('time')
x_exp = obs_1d.get_history('expected_position')[:, 0]
px_exp = obs_1d.get_history('expected_momentum')[:, 0]
E_exp = obs_1d.get_history('expected_energy')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(x, rho_1d_init, 'k--', alpha=0.6, label='t = 0')
axes[0].plot(x, rho_1d_final, 'b-', lw=2, label='t = 3.6')
axes[0].axvline(0.0, color='r', ls=':', label='Centro Coulomb (x=0)')
axes[0].set_xlabel('x')
axes[0].set_ylabel(r'$\rho(x)$')
axes[0].set_title('Densidad de Probabilidad 1D')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_hist, x_exp, 'b-', lw=2, label=r'$\langle x \rangle(t)$')
axes[1].plot(t_hist, px_exp, 'g--', lw=2, label=r'$\langle p_x \rangle(t)$')
axes[1].set_xlabel('Tiempo t')
axes[1].set_title('Posición y Momento Esperados')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_hist, E_exp, 'crimson', lw=2)
axes[2].set_xlabel('Tiempo t')
axes[2].set_ylabel(r'$\langle H \rangle$')
axes[2].set_title('Energía Esperada Total')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Dispersión 2D en un Potencial de Coulomb y Corriente de Probabilidad $\mathbf{J}$

Ahora simulamos un paquete 2D con parámetro de impacto $y_0 = -1.2$ que pasa cerca del núcleo atractivo en $(0, 0)$.

In [ ]:
grid_2d = Grid(shape=(80, 80), spacing=(0.14, 0.14))
coulomb_2d = CoulombPotential(Z=3.0, epsilon=0.4)

spinor_2d = ConstantSpinor(momentum=[3.5, 0.5, 0.0])
initial_2d = GaussianPacket(spinor_2d, center=[-2.5, -1.2, 0.0], spatial_width=0.85)

problem_2d = (
    DiracProblemBuilder()
    .set_grid(grid_2d)
    .set_initial_state(initial_2d)
    .set_potential(coulomb_2d)
    .set_boundary_condition(_core.AbsorbingBoundary(strength=0.35, width=10))
    .set_time_parameters(time_step=0.005, total_time=1.4)
    .build()
)

solver_2d = DiracSolver(problem_2d)
obs_2d = Observables(solver_2d)
solver_2d.run_simulation()

psi_final = solver_2d.get_psi()
rho_2d = obs_2d.probability_density(psi_final).reshape(grid_2d.shape)
curr_2d = obs_2d.probability_current(psi_final)
jx_2d = curr_2d[:, 0].reshape(grid_2d.shape)
jy_2d = curr_2d[:, 1].reshape(grid_2d.shape)

x2 = grid_2d.coords[:, 0, 0]
y2 = grid_2d.coords[0, :, 1]
X2, Y2 = np.meshgrid(x2, y2, indexing='ij')

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(rho_2d.T, origin='lower', extent=[x2[0], x2[-1], y2[0], y2[-1]], cmap='inferno')
plt.colorbar(im, ax=ax, label=r'Densidad de Probabilidad $\rho(x,y)$')

skip = 4
ax.quiver(X2[::skip, ::skip], Y2[::skip, ::skip],
          jx_2d[::skip, ::skip], jy_2d[::skip, ::skip],
          color='cyan', pivot='mid', alpha=0.85)
ax.plot(0, 0, 'w*', markersize=14, label='Centro Coulomb (0,0)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Dispersión de Coulomb 2D: Densidad y Corriente $\mathbf{J}(x,y)$')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()